# BioEmu-1 re-sampling on a free GPU

Samples 100 conformers per protein with the current **bioemu-v1.2** checkpoint for the 51 proteins in
`resample_list.csv` (the 41 clean disordered entries of the study, the three worked examples and eight folded
controls). Output is written per protein as soon as it finishes, so a session timeout loses nothing: run all
cells again and finished proteins are skipped.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough). Expected total time: 2–4 h on a T4.
On Kaggle, set the accelerator to GPU and the output lands in `/kaggle/working/bioemu_resample`.

In [ ]:
import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = pathlib.Path("/content/drive/MyDrive/bioemu_resample")
else:
    OUT = pathlib.Path("/kaggle/working/bioemu_resample") if pathlib.Path("/kaggle").exists() else pathlib.Path("bioemu_resample")
OUT.mkdir(parents=True, exist_ok=True)
print("output folder:", OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install -q "bioemu[cuda]" pandas

In [ ]:
import pandas as pd, requests, io
URL = "https://raw.githubusercontent.com/Natalija-Stepurko/bioemu-saxs-campaign/main/notebooks/resample_list.csv"
proteins = pd.read_csv(io.StringIO(requests.get(URL, timeout=60).text))
proteins = proteins.sort_values("length")           # short ones first, so progress is visible early
print(len(proteins), "proteins,", proteins.length.sum(), "residues")
proteins.head()

In [ ]:
import json, time, shutil, traceback
from bioemu.sample import main as sample

MODEL = "bioemu-v1.2"
N = 100
log = OUT / "progress.json"
done = json.load(open(log)) if log.exists() else {}
for _, r in proteins.iterrows():
    lab = r["label"]
    if lab in done and done[lab].get("ok"):
        continue
    t0 = time.time()
    tmp = OUT / f"_tmp_{lab}"
    try:
        sample(sequence=r["sequence"], num_samples=N, output_dir=str(tmp), model_name=MODEL, filter_samples=True)
        dest = OUT / lab
        if dest.exists():
            shutil.rmtree(dest)
        tmp.rename(dest)
        n_files = len(list(dest.glob("*")))
        done[lab] = {"ok": True, "seconds": round(time.time() - t0), "length": int(r["length"]), "files": n_files,
                     "model": MODEL, "num_samples": N}
        print(f"{lab}: {int(r['length'])} aa, {done[lab]['seconds']} s, {n_files} files")
    except Exception as e:
        done[lab] = {"ok": False, "error": repr(e)[:300], "model": MODEL}
        print(f"{lab}: FAILED {e!r}")
        traceback.print_exc()
    json.dump(done, open(log, "w"), indent=1)
print("finished:", sum(1 for v in done.values() if v.get("ok")), "ok;", sum(1 for v in done.values() if not v.get("ok")), "failed")

In [ ]:
# zip for download (Colab: also appears in Drive as bioemu_resample.zip)
import shutil
zip_path = shutil.make_archive(str(OUT.parent / "bioemu_resample"), "zip", OUT)
print(zip_path)
if IN_COLAB:
    from google.colab import files
    # files.download(zip_path)   # uncomment to download directly to your computer